# Búsqueda Voraz Primero el Mejor (Greedy Best-First Search)

## Descripción de la actividad

En esta práctica se estudiará el funcionamiento de **Greedy Best-First Search (GBFS)** a partir de la estructura general de búsqueda desarrollada previamente para algoritmos no informados.

El punto de partida será una implementación de **Depth-First Search (DFS)** basada en búsqueda en árboles. A partir de ella se realizarán modificaciones progresivas para:

1. Convertir el algoritmo de **búsqueda en árboles** en un algoritmo de **búsqueda en grafos**.
2. Generalizar el manejo de la frontera mediante una **función de evaluación** $f(n)$.
3. Implementar **Greedy Best-First Search**, utilizando:

$f(n)=h(n)$,

donde $h(n)$ representa una estimación del costo restante desde el nodo $n$ hasta el objetivo.
4. Utilizar la **distancia Manhattan** como función heurística:

$h(n)=|x_n-x_G|+|y_n-y_G|$.

5. Registrar el tamaño de la frontera durante la ejecución para comparar experimentalmente el comportamiento de **DFS** y **Greedy Best-First Search**.

El problema se desarrollará sobre un entorno bidimensional tipo **GridWorld**. El agente deberá desplazarse desde un estado inicial hasta un estado objetivo mediante movimientos en las cuatro direcciones cardinales:

- izquierda,
- derecha,
- arriba,
- abajo.

Como reto final, se extenderá el problema para incorporar **celdas no transitables u obstáculos**, identificando qué componente de la formulación del problema debe modificarse para representar correctamente este nuevo escenario.

---

**Asignatura:** Inteligencia Artificial (0406)  
**Grupo:** 01  
**Semestre:** 2027-1  
**Fecha de entrega:** 20 de octubre de 2026

**Alumno:** Carlos Roberto Vázquez Villegas

## 1. Definición del escenario

Retomaremos el mismo entorno **GridWorld** utilizado en la actividad anterior. El agente se desplaza sobre una cuadrícula de $3\times3$, iniciando en la posición $(1,1)$ y buscando alcanzar el estado objetivo $(3,3)$.

En esta primera etapa **no se modificarán las características del problema**. Esto permitirá comparar distintas estrategias de búsqueda sobre exactamente el mismo espacio de estados.

Ejecute la siguiente celda para definir los parámetros básicos del entorno.

In [1]:
# Parámetros del entorno GridWorld

# Tamaño de la cuadrícula
N = 3

# Acciones disponibles para el agente
# Cada acción corresponde a un desplazamiento (dx, dy)
acciones = {
    "izquierda": (-1, 0),
    "abajo": (0, -1),
    "arriba": (0, 1),
    "derecha": (1, 0)
}

# Estado inicial del agente
estado_inicial = (1, 1)

# Estado objetivo
estado_objetivo = (N, N)

## Definición del problema

El problema de búsqueda se describe mediante:

- una **función sucesora**, que determina las acciones válidas y los estados alcanzables;
- una **prueba de objetivo**, que verifica si se ha alcanzado la meta;
- una **función de costo**, que asigna un costo a cada transición.

En este GridWorld:

$
\text{SUCESOR-FN}(s)=\{(a,s')\},
\qquad
c(s,a,s')=1.
$

Por tanto, el costo acumulado $g(n)$ coincide con el número de movimientos realizados.

In [2]:
# Funciones del problema
def es_valido(x, y):
    return 1 <= x <= N and 1 <= y <= N

def sucesor_fn(estado):
    x, y = estado
    sucesores = []

    for accion, (dx, dy) in acciones.items():
        nuevo_x = x + dx
        nuevo_y = y + dy

        if es_valido(nuevo_x, nuevo_y):
            sucesores.append((accion, (nuevo_x, nuevo_y)))

    return sucesores

def test_objetivo(estado):
    return estado == estado_objetivo

def costo_accion(estado, accion, estado_sucesor):
    return 1

## Problema de búsqueda

Un problema de búsqueda puede representarse mediante cuatro componentes:

1. **Estado inicial** $s_0$.
2. **Función sucesora** $\text{SUCESOR-FN}(s)$.
3. **Prueba de objetivo**.
4. **Función de costo** $c(s,a,s')$.

A partir de estos elementos, el algoritmo explora el espacio de estados para encontrar un camino desde $s_0$ hasta un estado objetivo.

En esta implementación agruparemos estos componentes en una sola estructura.

In [3]:
# Definición del problema
problema = {
    "estado_inicial": estado_inicial,
    "funcion_sucesora": sucesor_fn,
    "test_objetivo": test_objetivo,
    "costo_accion": costo_accion
}

## Representación de nodos

Es importante distinguir entre un **estado** del problema y un **nodo de búsqueda**.

Un estado representa una configuración del entorno, mientras que un nodo almacena además información sobre el camino seguido para alcanzarlo.

Cada nodo contendrá:

- **estado**: estado representado;
- **padre**: nodo desde el cual fue generado;
- **acción**: acción aplicada;
- **costo_camino**: costo acumulado $g(n)$;
- **profundidad**: número de transiciones desde el nodo inicial.

Esta información permitirá reconstruir posteriormente la solución encontrada.

In [4]:
def hacer_nodo(estado, padre, accion, costo_camino, profundidad):
    return {
        "estado": estado,
        "padre": padre,
        "accion": accion,
        "costo_camino": costo_camino,
        "profundidad": profundidad
    }

## Operaciones sobre los nodos

Durante la búsqueda se realizan dos operaciones fundamentales sobre los nodos:

- **Expandir un nodo:** utiliza la función sucesora para generar nuevos nodos a partir de las transiciones válidas.
- **Reconstruir la solución:** sigue las referencias al nodo padre desde el objetivo hasta el nodo inicial.

Al expandir un nodo, el costo acumulado y la profundidad se actualizan como:

$g(n') = g(n) + c(s,a,s')$

$d(n') = d(n) + 1$

La referencia al nodo padre permite reconstruir posteriormente la secuencia de acciones que forma la solución.

In [5]:
def expandir(nodo, problema):
    sucesores = []

    for accion, estado in problema["funcion_sucesora"](nodo["estado"]):

        nuevo_nodo = hacer_nodo(
            estado,
            nodo,
            accion,
            nodo["costo_camino"] + problema["costo_accion"](
                nodo["estado"],
                accion,
                estado
            ),
            nodo["profundidad"] + 1
        )

        sucesores.append(nuevo_nodo)

    return sucesores


def solucion(nodo):
    camino = []

    while nodo["padre"] is not None:
        camino.append(
            (nodo["padre"]["estado"], nodo["accion"])
        )
        nodo = nodo["padre"]

    camino.reverse()
    return camino

## Búsqueda en árboles

El algoritmo mantiene una **frontera** con los nodos generados que aún no han sido expandidos.

En cada iteración:

1. extrae un nodo de la frontera;
2. verifica si corresponde al objetivo;
3. si no es así, lo expande y agrega sus sucesores.

La estrategia de búsqueda depende de cómo se administra la frontera: FIFO para BFS, LIFO para DFS o prioridad por costo para UCS.

En esta versión se utiliza **búsqueda en árboles**, por lo que un mismo estado puede generarse más de una vez. Posteriormente se modificará el algoritmo para realizar **búsqueda en grafos**.

In [6]:
# Algoritmo general de búsqueda en árboles
def busqueda_arbol(problema, frontera):

    frontera.put([
        hacer_nodo(
            problema["estado_inicial"],
            None,
            None,
            0,
            0
        )
    ])

    while frontera.pendientes():

        nodo = frontera.get()

        if problema["test_objetivo"](nodo["estado"]):
            return solucion(nodo)

        frontera.put(expandir(nodo, problema))

    return None

## Actividad 1 — DFS como punto de partida

La estrategia de búsqueda depende de cómo se administra la **frontera**.

En **Depth-First Search (DFS)**, la frontera se comporta como una pila **LIFO** (*Last-In, First-Out*): el último nodo insertado es el primero en extraerse.

### Instrucciones

1. Implemente una clase `FronteraDFS` utilizando una lista de Python.
2. Mantenga la interfaz:
   - `pendientes()`
   - `get()`
   - `put(data)`
3. Ejecute DFS utilizando el algoritmo `busqueda_arbol`.

```python
f = FronteraDFS()
sol = busqueda_arbol(problema, f)
list(sol)
```

Observe el comportamiento de la búsqueda y la solución obtenida. Esta implementación servirá como punto de partida para modificar posteriormente el algoritmo de **búsqueda en árboles** a **búsqueda en grafos**.

In [7]:
# Frontera para Depth-First Search (DFS) usando comportamiento LIFO (pila)
class FronteraDFS:

    def __init__(self):
        self.pila = []

    def pendientes(self):
        return len(self.pila)

    def get(self):
        return self.pila.pop()

    def put(self, nodos):
        for nodo in nodos:
            self.pila.append(nodo)

## Actividad 2 — Implementar búsqueda en grafos

La implementación actual realiza **búsqueda en árboles**, por lo que un mismo estado puede aparecer y expandirse más de una vez.

Modifique el algoritmo para implementar **búsqueda en grafos**, siguiendo lo más cercanamente posible el siguiente procedimiento:

1. Inicializar:

$cerrado \leftarrow \emptyset$

2. Insertar en la frontera un nodo correspondiente al estado inicial.

3. Mientras la frontera no esté vacía:

   - extraer un nodo de la frontera;
   - aplicar el **test objetivo**;
   - si el estado del nodo no pertenece a `cerrado`:
     - agregar su estado a `cerrado`;
     - expandir el nodo;
     - insertar todos sus sucesores en la frontera.

4. Si la frontera queda vacía, devolver fallo.

### Instrucciones

- Implemente una nueva función llamada `busqueda_grafo`.
- Utilice un `set` de Python para representar `cerrado`.
- Mantenga sin cambios la clase `FronteraDFS`.
- Ejecute DFS utilizando primero `busqueda_arbol` y después `busqueda_grafo`.

### Comparación

Observe ambas ejecuciones y responda:

1. ¿Ambas encuentran una solución?
<br>**Sí, tanto la búsqueda en árboles como la búsqueda en grafos logran encontrar una ruta válida hacia el estado objetivo en este escenario finito.**

2. ¿La solución obtenida es la misma?
<br>**Sí, en este GridWorld inicial de $3 \times 3$ sin obstáculos, ambas estrategias de búsqueda retornan una solución idéntica con un costo óptimo de 4 movimientos (por ejemplo: derecha, derecha, arriba, arriba).**

3. ¿Qué ocurre con los estados que pueden alcanzarse por más de un camino?
<br>**En la búsqueda en árboles, estos estados se visitan, se vuelven a colocar en la frontera y se expanden repetidamente cada vez que se llega a ellos, causando un procesamiento redundante. En contraste, en la búsqueda en grafos, la estructura condicional revisa el conjunto `cerrado`, lo que permite descartar e ignorar automáticamente cualquier camino redundante que llegue a un estado que ya fue explorado.**

4. ¿Qué ventaja introduce el conjunto `cerrado`?
<br>**Introduce una "memoria" que registra los estados ya visitados, lo cual aporta dos grandes ventajas: primero, evita que el algoritmo caiga en bucles infinitos al explorar grafos con ciclos; segundo, reduce drásticamente el tiempo de ejecución y el tamaño de la frontera al podar ramas redundantes, evitando que el algoritmo recalcule expansiones innecesarias.**

In [8]:
# Implementación de la función busqueda_grafo en base al pseudocódigo: Búsqueda grafos
def busqueda_grafo(problema, frontera):
    cerrado = set()  # Línea 1
    
    # Línea 2: Crear e insertar el nodo inicial
    nodo_inicial = hacer_nodo(problema["estado_inicial"], None, None, 0, 0)
    frontera.put([nodo_inicial])
    
    # Línea 3: while frontera no vacía
    while frontera.pendientes():
        # Línea 4: Extraer nodo
        nodo = frontera.get()
        
        # Línea 5 y 6: Test objetivo
        if problema["test_objetivo"](nodo["estado"]):
            return solucion(nodo)
            
        # Línea 7: Verificar si el estado no está en cerrado
        if nodo["estado"] not in cerrado:
            # Línea 8: Añadir a cerrado
            cerrado.add(nodo["estado"])
            # Línea 9: Expandir e insertar sucesores
            sucesores = expandir(nodo, problema)
            frontera.put(sucesores)
            
    # Línea 10
    return None

In [9]:
## VALIDA TU IMPLEMENTACIÓN

f = FronteraDFS()
sol = busqueda_grafo(problema, f)

resultado_esperado = [
    ((1, 1), "derecha"),
    ((2, 1), "derecha"),
    ((3, 1), "arriba"),
    ((3, 2), "arriba")
]

assert sol == resultado_esperado, "La solución obtenida no coincide con la esperada."

print("La búsqueda en grafos funciona correctamente.")

La búsqueda en grafos funciona correctamente.


## Actividad 3 — Generalizar la frontera mediante $f(n)$

En las estrategias de búsqueda con prioridad, el siguiente nodo que se extrae de la frontera depende de una **función de evaluación** $f(n)$.

Por ejemplo:

$f_{\text{UCS}}(n)=g(n)$

donde $g(n)$ es el costo acumulado del camino hasta el nodo.

### Instrucciones

1. Modifique la implementación de la frontera con prioridad para que reciba una función $f(n)$.
2. Al insertar cada nodo, calcule su prioridad mediante:

$\text{prioridad}=f(n)$

3. La frontera debe extraer siempre el nodo con el menor valor de $f(n)$.
4. Mantenga la misma interfaz utilizada anteriormente:
   - `pendientes()`
   - `get()`
   - `put(nodos)`

5. Verifique inicialmente el funcionamiento definiendo:

$f(n)=g(n)$

de modo que la estrategia obtenida sea equivalente a **Uniform Cost Search (UCS)**.

Esta generalización permitirá reutilizar la misma frontera con diferentes funciones de evaluación en las siguientes actividades.

In [10]:
import heapq

class FronteraPrioridad:
    def __init__(self, f):
        self.heap = []
        self.f = f
        self.contador = 0

    def pendientes(self):
        return len(self.heap) > 0

    def get(self):
        # heapq.heappop devuelve la tupla (prioridad, contador, nodo)
        # Solo nos interesa regresar el nodo
        _, _, nodo = heapq.heappop(self.heap)
        return nodo

    def put(self, nodos):            
        for nodo in nodos:
            prioridad = self.f(nodo)
            # Insertamos una tupla con la prioridad, el desempate y el nodo
            heapq.heappush(self.heap, (prioridad, self.contador, nodo))
            self.contador += 1

In [11]:
def f_ucs(nodo):
    return nodo["costo_camino"]

f = FronteraPrioridad(f_ucs)
sol = busqueda_grafo(problema, f)

list(sol)

[((1, 1), 'arriba'),
 ((1, 2), 'arriba'),
 ((1, 3), 'derecha'),
 ((2, 3), 'derecha')]

## Actividad 4 — Implementar Greedy Best-First Search

En **Greedy Best-First Search (GBFS)**, la prioridad de cada nodo se determina únicamente mediante una función heurística:

$f(n)=h(n)$

Para este GridWorld utilizaremos como heurística la **distancia Manhattan** entre el estado actual $(x,y)$ y el estado objetivo $(x_G,y_G)$:

$h(n)=|x-x_G|+|y-y_G|$

### Instrucciones

1. Implemente una función `h(nodo)` que calcule la distancia Manhattan desde el estado del nodo hasta el objetivo.
2. Defina la función de evaluación de Greedy como:

$f_{\text{GBFS}}(n)=h(n)$

3. Utilice la clase `FronteraPrioridad` desarrollada en la actividad anterior.
4. Ejecute `busqueda_grafo` utilizando esta nueva función de evaluación.
5. Observe el orden en que GBFS orienta la búsqueda hacia el estado objetivo.

### Reflexión

- ¿Qué información utiliza GBFS para seleccionar el siguiente nodo?
<br>**Utiliza únicamente la estimación del costo restante hasta la meta dada por la heurística $h(n)$.**
- ¿Considera el costo acumulado $g(n)$?
<br>**No, ignora por completo el costo recorrido desde el inicio.**
- ¿Qué diferencia conceptual existe entre $f_{\text{UCS}}(n)=g(n)$ y $f_{\text{GBFS}}(n)=h(n)$?
<br>**$f_{\text{UCS}}(n)$ mira el costo pasado para asegurar el camino más corto, mientras que $f_{\text{GBFS}}(n)$ mira el costo futuro estimado para llegar lo más rápido posible, aunque no garantiza el camino óptimo.**

In [12]:
def h(nodo):
    x, y = nodo["estado"]
    x_G, y_G = estado_objetivo
    
    # Distancia Manhattan: |x - x_G| + |y - y_G|
    return abs(x - x_G) + abs(y - y_G)

In [13]:
def f_gbfs(nodo):
    return h(nodo)

In [14]:
# VALIDA TU IMPLEMENTACIÓN

f = FronteraPrioridad(f_gbfs)
sol = busqueda_grafo(problema, f)

resultado_esperado = [
    ((1, 1), "arriba"),
    ((1, 2), "arriba"),
    ((1, 3), "derecha"),
    ((2, 3), "derecha")
]

assert sol == resultado_esperado, "La solución obtenida no coincide con la esperada."

print("Greedy Best-First Search funciona correctamente.")

Greedy Best-First Search funciona correctamente.


## Actividad 5 — Registrar el tamaño de la frontera

Además de encontrar una solución, podemos instrumentar un algoritmo para observar su comportamiento durante la ejecución.

En esta actividad registraremos el número de nodos almacenados en la **frontera**:

$|\text{frontera}|$

### Instrucciones

1. Modifique `FronteraDFS` y `FronteraPrioridad` para incorporar una lista llamada `tamanos`.
2. Después de cada operación `get()` o `put()`, registre el número de nodos pendientes en la frontera.
3. Ejecute `busqueda_grafo` utilizando:
   - DFS;
   - Greedy Best-First Search.
4. Muestre las secuencias almacenadas en `tamanos`.

### Reflexión

- ¿Qué representa cada valor registrado?
<br>**Representa la cantidad exacta de nodos almacenados en la frontera en ese instante específico de la ejecución (después de insertar o extraer).**
- ¿Por qué el tamaño de la frontera aumenta y disminuye durante la búsqueda?
<br>**Aumenta cuando el algoritmo expande un nodo e inserta sus sucesores, y disminuye cuando se extrae un nodo para evaluarlo.**
- ¿El tamaño de la frontera representa por sí solo toda la memoria utilizada por una búsqueda en grafos? ¿Qué otra estructura utiliza el algoritmo?
<br>**No, no representa toda la memoria. En la búsqueda en grafos, el algoritmo también utiliza el conjunto `cerrado` para almacenar los estados ya explorados y evitar reevaluarlos.**

In [15]:
class FronteraDFS:
    def __init__(self):
        self.pila = []
        self.tamanos = []

    def pendientes(self):
        return len(self.pila) > 0

    def get(self):
        nodo = self.pila.pop()
        self.tamanos.append(len(self.pila))
        return nodo

    def put(self, nodos):
        self.pila.extend(nodos)
        self.tamanos.append(len(self.pila))

In [16]:
import heapq

class FronteraPrioridad:
    def __init__(self, f):
        self.heap = []
        self.f = f
        self.contador = 0
        self.tamanos = []

    def pendientes(self):
        return len(self.heap) > 0

    def get(self):
        _, _, nodo = heapq.heappop(self.heap)
        self.tamanos.append(len(self.heap))
        return nodo

    def put(self, nodos):
        if not isinstance(nodos, list):
            nodos = [nodos]
            
        for nodo in nodos:
            prioridad = self.f(nodo)
            heapq.heappush(self.heap, (prioridad, self.contador, nodo))
            self.contador += 1
        self.tamanos.append(len(self.heap))

In [17]:
# DFS
frontera_dfs = FronteraDFS()
sol_dfs = busqueda_grafo(problema, frontera_dfs)

# GBFS
frontera_gbfs = FronteraPrioridad(f_gbfs)
sol_gbfs = busqueda_grafo(problema, frontera_gbfs)

In [18]:
print("DFS :", frontera_dfs.tamanos)
print("GBFS:", frontera_gbfs.tamanos)

DFS : [1, 0, 2, 1, 4, 3, 5, 4, 7, 6]
GBFS: [1, 0, 2, 1, 4, 3, 5, 4, 7, 6]


### Observación

En este escenario pequeño, DFS y GBFS pueden registrar una evolución similar del tamaño de la frontera.

Esto no significa que ambas estrategias sean equivalentes. En la siguiente actividad modificaremos el escenario para observar con mayor claridad las diferencias en su comportamiento.

## Actividad 6 — Comparación experimental de DFS y GBFS

En el GridWorld anterior, DFS y GBFS pueden presentar un comportamiento similar debido al tamaño reducido del problema y al orden en que se generan los sucesores.

Para observar con mayor claridad las diferencias entre ambas estrategias, utilizaremos ahora:

$N=8$

y modificaremos el orden de las acciones. El mismo escenario y el mismo orden de sucesores deberán utilizarse para ambos algoritmos.

### Instrumentación de la búsqueda

Modifique la búsqueda en grafos para registrar, después de cada expansión:

- número de estados expandidos;
- tamaño de la frontera;
- tamaño del conjunto `cerrado`.

Defina además:

$M(t)=|\text{frontera}_t|+|\text{cerrado}_t|$

como una medida experimental del número de elementos almacenados durante la búsqueda.

Ejecute el experimento utilizando:

- DFS;
- Greedy Best-First Search.

Compare para ambas estrategias:

1. número de estados expandidos;
2. longitud de la solución encontrada;
3. tamaño máximo de la frontera;
4. tamaño máximo de `cerrado`;
5. máximo valor observado de $M(t)$.

### Relación con la complejidad espacial

Las cotas estudiadas inicialmente para **búsqueda en árboles** dependen de la estructura del árbol generado. Por ejemplo:

$\text{DFS}=O(bm)$

mientras que GBFS puede requerir, en el peor caso:

$O(b^m)$,

donde $b$ es el factor de ramificación y $m$ la profundidad máxima.

En esta práctica utilizamos **búsqueda en grafos**, por lo que el algoritmo mantiene además un conjunto `cerrado` con los estados ya explorados.

En este caso, resulta más natural expresar el uso de memoria en función del número de estados alcanzables:

$O(|V|)$,

donde $V$ representa el conjunto de estados del espacio de búsqueda.

Para un GridWorld de tamaño $N\times N$, sin obstáculos:

$|V|=N^2$.

Por tanto, para:

$N=8$,

el espacio contiene como máximo:

$|V|=64$

estados diferentes.

Aunque DFS y GBFS tienen la misma cota espacial en términos de $|V|$ para este escenario, esto no significa que almacenen o exploren la misma cantidad de información en una ejecución concreta.

### Reflexión

1. ¿Cuántos de los 64 estados posibles fueron expandidos por DFS y por GBFS?
<br>**DFS expandió 56 de los 64 estados posibles, mientras que GBFS expandió únicamente 14.**

2. ¿Cuál estrategia alcanzó un mayor tamaño de frontera?
<br>**DFS alcanzó un tamaño máximo de frontera considerablemente mayor (91) en comparación con GBFS (27).**

3. ¿Cuál presentó un mayor valor de $M(t)$?
<br>**DFS presentó un valor de $M(t)$ significativamente mayor, alcanzando un máximo de 147 frente a los 41 de GBFS.**

4. Si ambas estrategias tienen una cota espacial $O(|V|)$, ¿por qué los valores observados son diferentes?
<br>**La cota espacial $O(|V|)$ describe el límite teórico en el peor de los casos. En la práctica, los valores difieren porque la heurística permitió a GBFS encontrar la meta explorando solo una pequeña porción del espacio, mientras que la estrategia ciega de DFS lo llevó a recorrer casi toda la cuadrícula antes de terminar.**

5. ¿Qué efecto tuvo la heurística sobre el número de estados expandidos por GBFS?
<br>**La heurística redujo los estados expandidos en un 75% (de 56 a 14) al dirigir la búsqueda casi en línea recta hacia la meta, evitando la exploración exhaustiva de caminos innecesarios.**

6. ¿Podemos concluir, a partir de una sola ejecución, que GBFS siempre utilizará menos memoria que DFS? Justifique.
<br>**No. En escenarios donde existen obstáculos como "trampas en forma de herradura" o grandes callejones sin salida, la heurística engaña a GBFS guiándolo directo hacia el fondo del bloqueo porque matemáticamente parece acercarse a la meta. Al quedar atrapado en ese mínimo local, el algoritmo se ve obligado a explorar y almacenar en memoria casi todas las celdas del interior de la trampa antes de poder retroceder y rodear el obstáculo, lo que puede provocar un consumo de memoria igual o incluso superior al de DFS.**

In [19]:
def busqueda_grafo_medida(problema, frontera):
    cerrado = set()
    expandidos = 0
    tamanos_frontera = []
    tamanos_cerrado = []
    memoria_total = []

    nodo_inicial = hacer_nodo(
        problema["estado_inicial"],
        None,
        None,
        0,
        0
    )
    frontera.put([nodo_inicial])

    while frontera.pendientes():
        # Medir tamaños antes de extraer
        tf = len(frontera.pila) if hasattr(frontera, 'pila') else len(frontera.heap)
        tc = len(cerrado)
        
        tamanos_frontera.append(tf)
        tamanos_cerrado.append(tc)
        memoria_total.append(tf + tc)

        nodo = frontera.get()

        if problema["test_objetivo"](nodo["estado"]):
            return {
                "solucion": solucion(nodo),
                "expandidos": expandidos,
                "tamanos_frontera": tamanos_frontera,
                "tamanos_cerrado": tamanos_cerrado,
                "memoria_total": memoria_total
            }

        if nodo["estado"] not in cerrado:
            cerrado.add(nodo["estado"])
            expandidos += 1  # Solo contamos cuando realmente lo expandimos
            sucesores = expandir(nodo, problema)
            frontera.put(sucesores)

    return None

In [20]:
N = 8

# El orden de las acciones se modifica para observar
# diferencias más claras entre DFS y GBFS
acciones = {
    "derecha": (1, 0),
    "arriba": (0, 1),
    "izquierda": (-1, 0),
    "abajo": (0, -1)
}

estado_inicial = (1, 1)
estado_objetivo = (N, N)

problema = {
    "estado_inicial": estado_inicial,
    "funcion_sucesora": sucesor_fn,
    "test_objetivo": test_objetivo,
    "costo_accion": costo_accion
}

In [21]:
# DFS
frontera_dfs = FronteraDFS()

resultado_dfs = busqueda_grafo_medida(
    problema,
    frontera_dfs
)


# GBFS
frontera_gbfs = FronteraPrioridad(f_gbfs)

resultado_gbfs = busqueda_grafo_medida(
    problema,
    frontera_gbfs
)

In [22]:
print("DFS")
print("Nodos expandidos:", resultado_dfs["expandidos"])
print("Longitud solución:", len(resultado_dfs["solucion"]))
print("Máximo frontera:", max(resultado_dfs["tamanos_frontera"]))
print("Máximo cerrado:", max(resultado_dfs["tamanos_cerrado"]))
print("Máximo M(t):", max(resultado_dfs["memoria_total"]))

print()

print("GBFS")
print("Nodos expandidos:", resultado_gbfs["expandidos"])
print("Longitud solución:", len(resultado_gbfs["solucion"]))
print("Máximo frontera:", max(resultado_gbfs["tamanos_frontera"]))
print("Máximo cerrado:", max(resultado_gbfs["tamanos_cerrado"]))
print("Máximo M(t):", max(resultado_gbfs["memoria_total"]))

DFS
Nodos expandidos: 56
Longitud solución: 56
Máximo frontera: 91
Máximo cerrado: 56
Máximo M(t): 147

GBFS
Nodos expandidos: 14
Longitud solución: 14
Máximo frontera: 27
Máximo cerrado: 14
Máximo M(t): 41


## Reto final — GridWorld con obstáculos

Hasta ahora hemos trabajado con una cuadrícula sin restricciones internas. Considere ahora un escenario en el que algunas celdas representan **obstáculos** y no pueden ser atravesadas.

Se proporciona una función que genera automáticamente un GridWorld con:

- dimensiones variables;
- estado inicial;
- estado objetivo;
- conjunto de obstáculos.

### Reto

Adapte la implementación desarrollada durante la práctica utilizando el **menor número posible de cambios**, de forma que DFS y GBFS puedan resolver este nuevo escenario.

Identifique qué componentes deben modificarse y cuáles pueden mantenerse sin cambios.

En particular, determine:

- qué parte de la definición del problema debe considerar los obstáculos;
- si es necesario modificar `busqueda_grafo`;
- si es necesario modificar `FronteraDFS`;
- si es necesario modificar `FronteraPrioridad`;
- si puede mantenerse la heurística Manhattan.

### Reflexión

¿Qué componente de la formulación del problema representa las restricciones impuestas por los obstáculos?
<br>**El componente formal que representa las restricciones impuestas por los obstáculos es la función sucesora (o modelo de transición). En la representación formal de un problema de búsqueda en el espacio de estados, esta función es la encargada exclusiva de definir las reglas del entorno, determinando qué acciones son válidas y generando los estados legalmente alcanzables a partir de la posición actual del agente.**
<br>
<br>**Para introducir las celdas no transitables en el entorno GridWorld, no es necesario modificar el núcleo matemático de los algoritmos de búsqueda (como DFS o GBFS) ni las estructuras de datos de sus fronteras. La adaptación recae puramente en el mecanismo de validación de coordenadas que es invocada internamente por la función sucesora.**
<br>
<br>**Al evaluar si la coordenada destino pertenece al conjunto de muros, la función sucesora actúa como un filtro absoluto. Si la celda es un obstáculo, el sistema simplemente descarta esa transición y no genera el nodo sucesor, impidiendo que el algoritmo intente avanzar por ese camino. De esta forma, todas las restricciones físicas y topológicas del escenario se modelan lógicamente durante la fase de expansión, manteniendo el algoritmo central de búsqueda completamente abstracto, general e intacto.**

In [43]:
### CREACIÓN DE ESCENARIOS ALEATORIOS

import random

def crear_escenario(ancho=10, alto=8, prob_muro=0.2, semilla=7, inicio=(1, 1)):
    random.seed(semilla)

    celdas = [
        (x, y)
        for x in range(1, ancho + 1)
        for y in range(1, alto + 1)
        if (x, y) != inicio
    ]

    muros = {
        celda
        for celda in celdas
        if random.random() < prob_muro
    }

    libres = [celda for celda in celdas if celda not in muros]
    meta = random.choice(libres)

    return {
        "ancho": ancho,
        "alto": alto,
        "muros": muros,
        "inicio": inicio,
        "meta": meta
    }

In [42]:
problema = crear_problema(
    ancho=10,
    alto=8,
    prob_muro=0.2,
    semilla=1984,
    inicio=(1, 1)
)

print("Inicio:", problema["estado_inicial"])
print("Meta:", problema["estado_objetivo"])
print("Obstáculos:", problema["muros"])

Inicio: (1, 1)
Meta: (5, 8)
Obstáculos: {(10, 5), (2, 8), (7, 7), (3, 1), (9, 6), (10, 1), (10, 7), (5, 7), (2, 3), (7, 6), (9, 8), (6, 6), (8, 2), (5, 2)}


In [41]:
## FUNCIÓN PARA MOSTRAR EL PROBLEMA A RESOLVER

def mostrar(problema, camino=None):
    camino = camino or []

    # La solución tiene pares (estado, acción)
    estados_camino = {estado for estado, accion in camino}

    for y in reversed(range(1, problema["alto"] + 1)):
        fila = ""

        for x in range(1, problema["ancho"] + 1):
            p = (x, y)

            if p == problema["estado_inicial"]:
                fila += "S "
            elif p == problema["estado_objetivo"]:
                fila += "G "
            elif p in problema["muros"]:
                fila += "# "
            elif p in estados_camino:
                fila += "o "
            else:
                fila += ". "

        print(fila)

    print()

In [40]:
mostrar(problema)

. # . . G . . . # . 
. . . . # . # . . # 
. . . . . # # . # . 
. . . . . . . . . # 
. . . . . . . . . . 
. # . . . . . . . . 
. . . . # . . # . . 
S . # . . . . . . # 



In [39]:
# FI UNAM
# Inteligencia Artificial
# Carlos Roberto Vázquez Villegas

def crear_problema(ancho=10, alto=8, prob_muro=0.2, semilla=7, inicio=(1, 1)):
    escenario = crear_escenario(ancho, alto, prob_muro, semilla, inicio)
    
    def es_valido_reto(x, y):
        # Ahora también verificamos que la celda no sea un muro
        return 1 <= x <= ancho and 1 <= y <= alto and (x, y) not in escenario["muros"]

    def sucesor_fn_reto(estado):
        x, y = estado
        sucesores = []
        for accion, (dx, dy) in acciones.items():
            nuevo_x, nuevo_y = x + dx, y + dy
            if es_valido_reto(nuevo_x, nuevo_y):
                sucesores.append((accion, (nuevo_x, nuevo_y)))
        return sucesores

    def test_objetivo_reto(estado):
        return estado == escenario["meta"]

    return {
        "estado_inicial": escenario["inicio"],
        "estado_objetivo": escenario["meta"],
        "funcion_sucesora": sucesor_fn_reto,
        "test_objetivo": test_objetivo_reto,
        "costo_accion": costo_accion,
        "ancho": escenario["ancho"],
        "alto": escenario["alto"],
        "muros": escenario["muros"]
    }

In [36]:
# Adaptamos la heurística para leer la meta desde el nuevo diccionario 'problema'
def h_reto(nodo):
    x, y = nodo["estado"]
    x_G, y_G = problema["estado_objetivo"]
    return abs(x - x_G) + abs(y - y_G)

def f_gbfs_reto(nodo):
    return h_reto(nodo)

# Ejecutamos las búsquedas
frontera_dfs_reto = FronteraDFS()
resultado_dfs_reto = busqueda_grafo_medida(problema, frontera_dfs_reto)

frontera_gbfs_reto = FronteraPrioridad(f_gbfs_reto)
resultado_gbfs_reto = busqueda_grafo_medida(problema, frontera_gbfs_reto)

In [38]:
print("--- DFS ---")
if resultado_dfs_reto:
    print("Nodos expandidos:", resultado_dfs_reto["expandidos"])
    print("Longitud solución:", len(resultado_dfs_reto["solucion"]))
else:
    print("DFS no encontró solución.")

print("\n--- GBFS ---")
if resultado_gbfs_reto:
    print("Nodos expandidos:", resultado_gbfs_reto["expandidos"])
    print("Longitud solución:", len(resultado_gbfs_reto["solucion"]))
    print("\nVisualización de la ruta de GBFS:")
    mostrar(problema, resultado_gbfs_reto["solucion"])
else:
    print("GBFS no encontró solución.")

--- DFS ---
Nodos expandidos: 57
Longitud solución: 53

--- GBFS ---
Nodos expandidos: 13
Longitud solución: 13

Visualización de la ruta de GBFS:
. # . o G . . . # . 
. . . o # . # . . # 
. . . o o # # . # . 
. . . . o . . . . # 
. . . . o . . . . . 
. # . o o . . . . . 
. o o o # . . # . . 
S o # . . . . . . # 

